### Volume / Row Count Check

In [ ]:
import os
import glob
import pandas as pd
# 1. Khai báo đường dẫn thư mục chứa dataset
folder_path = rf'D:\olist-customer-satisfaction-analysis\data\raw'

# 2. Lấy danh sách tất cả các file .csv trong thư mục
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

total_lines = 0

print(f"Đang đếm số dòng cho {len(csv_files)} files...\n")

for file in csv_files:
    # Mở file và đếm số dòng (dùng generator để không tốn RAM)
    with open(file, 'r', encoding='utf-8', errors='ignore') as f:
        # Trừ 1 để không tính dòng Header (tên cột)
        lines = sum(1 for _ in f) - 1 
        
    total_lines += lines
    print(f"{os.path.basename(file)}: {lines:,} dòng")

print("-" * 30)
print(f"=> Tổng cộng: {total_lines:,} dòng dữ liệu")

Đang đếm số dòng cho 9 files...

olist_customers_dataset.csv: 99,441 dòng
olist_geolocation_dataset.csv: 1,000,163 dòng
olist_orders_dataset.csv: 99,441 dòng
olist_order_items_dataset.csv: 112,650 dòng
olist_order_payments_dataset.csv: 103,886 dòng
olist_order_reviews_dataset.csv: 104,719 dòng
olist_products_dataset.csv: 32,951 dòng
olist_sellers_dataset.csv: 3,095 dòng
product_category_name_translation.csv: 71 dòng
------------------------------
=> Tổng cộng: 1,556,417 dòng dữ liệu


Nhận xét: Tất cả các table đều được import data đầy đủ. Riêng olist_order_reviews có sự khác biệt (104,719/99224). Nghi ngờ là do nội dung review có chứa xuống dòng

In [ ]:
path = r'D:\olist-customer-satisfaction-analysis\data\raw\olist_order_reviews_dataset.csv'

# Đọc không chia chunk, dùng python engine, bắt lỗi rõ ràng
df_full = pd.read_csv(
    path, 
    engine='python', 
    on_bad_lines='warn',   # sẽ in ra dòng nào bị lỗi parse
    encoding='utf-8'
)

print(f"Tổng số dòng đọc được: {len(df_full)}")

# Đếm số dòng thực tế trong file CSV gốc (kể cả header)
with open(path, 'r', encoding='utf-8', errors='replace') as f:
    raw_line_count = sum(1 for _ in f)
print(f"Số dòng raw trong file (ước lượng, có thể lệch do multiline field): {raw_line_count - 1}")

Tổng số dòng đọc được: 99224
Số dòng raw trong file (ước lượng, có thể lệch do multiline field): 104719


### Missing Value Check per Table


In [3]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import urllib

# --- Kết nối SQL Server bằng .env ---
load_dotenv()

server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')
driver = os.getenv('DB_DRIVER')

conn_str = urllib.parse.quote_plus(
    f'DRIVER={{{driver}}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'
)
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={conn_str}')

SCHEMA = 'staging'

# --- Bước 1: Lấy danh sách bảng + cột từ INFORMATION_SCHEMA ---
query_columns = text("""
    SELECT TABLE_NAME, COLUMN_NAME
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = :schema
    ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

with engine.connect() as conn:
    df_columns = pd.read_sql(query_columns, conn, params={'schema': SCHEMA})

# --- Bước 2: Với mỗi cột, đếm missing (NULL hoặc rỗng '') ---
results = []

with engine.connect() as conn:
    for table_name, col_name in zip(df_columns['TABLE_NAME'], df_columns['COLUMN_NAME']):
        query = text(f"""
            SELECT COUNT(*) AS missing_count
            FROM {SCHEMA}.[{table_name}]
            WHERE [{col_name}] IS NULL OR CAST([{col_name}] AS VARCHAR(MAX)) = ''
        """)
        try:
            missing_count = conn.execute(query).scalar()
        except Exception as e:
            missing_count = f'ERROR: {e}'
        
        results.append({
            'table_name': table_name,
            'column_name': col_name,
            'missing_count': missing_count
        })
        print(f'Đã kiểm tra: {table_name}.{col_name} -> missing: {missing_count}')

# --- Bước 3: Tổng hợp kết quả ---
df_missing = pd.DataFrame(results)

# Lấy thêm tổng số dòng mỗi bảng để tính % missing
query_rowcount = text("""
    SELECT t.name AS table_name, p.rows AS row_count
    FROM sys.tables t
    JOIN sys.partitions p ON t.object_id = p.object_id
    JOIN sys.schemas s ON t.schema_id = s.schema_id
    WHERE s.name = :schema AND p.index_id IN (0,1)
""")
with engine.connect() as conn:
    df_rowcount = pd.read_sql(query_rowcount, conn, params={'schema': SCHEMA})

df_missing = df_missing.merge(df_rowcount, on='table_name', how='left')
df_missing['missing_pct'] = (df_missing['missing_count'] / df_missing['row_count'] * 100).round(2)

# --- Bước 4: Xuất kết quả ---
pd.set_option('display.max_rows', None)
print(df_missing.sort_values(['table_name', 'column_name']).to_string(index=False))


Đã kiểm tra: olist_customers.customer_id -> missing: 0
Đã kiểm tra: olist_customers.customer_unique_id -> missing: 0
Đã kiểm tra: olist_customers.customer_zip_code_prefix -> missing: 0
Đã kiểm tra: olist_customers.customer_city -> missing: 0
Đã kiểm tra: olist_customers.customer_state -> missing: 0
Đã kiểm tra: olist_geolocation.geolocation_zip_code_prefix -> missing: 0
Đã kiểm tra: olist_geolocation.geolocation_lat -> missing: 0
Đã kiểm tra: olist_geolocation.geolocation_lng -> missing: 0
Đã kiểm tra: olist_geolocation.geolocation_city -> missing: 0
Đã kiểm tra: olist_geolocation.geolocation_state -> missing: 0
Đã kiểm tra: olist_order_items.order_id -> missing: 0
Đã kiểm tra: olist_order_items.order_item_id -> missing: 0
Đã kiểm tra: olist_order_items.product_id -> missing: 0
Đã kiểm tra: olist_order_items.seller_id -> missing: 0
Đã kiểm tra: olist_order_items.shipping_limit_date -> missing: 0
Đã kiểm tra: olist_order_items.price -> missing: 0
Đã kiểm tra: olist_order_items.freight_v

### Missing Values Check In Detail

-- 1. olist_orders: 160 missing values ở order_approved_at

-- 2. olist_orders: 1783 missing values ở order_delivered_carrier_date

-- 3. olist_orders: 2965 missing value ở order_delivered_customer_date

-- 4. olist_products: 610 missing values ở lần lượt các cột product_category_name, product_name_lenght, product_description_lenght, product_photos_qty

-- 5. olist_products: 2 missing values ở lần lượt các cột product_weight_g, product_length_cm, product_height_cm, product_width_cm


### Order Status of Rows Missing order_approved_at
Bi?u ?? ph?n b? `order_status` trong c?c ??n h?ng thi?u gi? tr? ? c?t `order_approved_at`.

In [ ]:
import matplotlib.pyplot as plt

orders_path = os.path.join(folder_path, 'olist_orders_dataset.csv')
orders = pd.read_csv(orders_path)

missing_approval_mask = orders['order_approved_at'].isna()
missing_orders = orders.loc[missing_approval_mask]
status_counts = missing_orders['order_status'].value_counts()

ax = status_counts.plot.pie(
    autopct='%1.1f%%',
    startangle=90,
    ylabel='',
    figsize=(8, 8),
)
ax.set_title(
    'Order Status for Orders Missing order_approved_at\n'
    f'(n={len(missing_orders):,})'
)
plt.tight_layout()
plt.show()

display(status_counts.rename('order_count').to_frame())